# Model Evaluation and Comparative Analysis
## Signature Memorization vs. Unseen Attack Generalization in Network Intrusion Detection

**Course:** ES 335 — Machine Learning, IIT Gandhinagar  
**Team Members:** Vedant & Dhruv  

---

### Research Objective
Investigate whether simpler machine learning models can generalize better to previously unseen attack types than more complex models in network intrusion detection, or whether complex models suffer from **signature memorization**.

### Experimental Controls
To guarantee fair, consistent, and reproducible comparison:
1. **Identical Feature Representation:** Canonical 41 raw features mapped to exactly 122 numerical features via `OneHotEncoder(handle_unknown='ignore')` + `StandardScaler()`.
2. **Zero Validation Leakage:** Encoders and scalers were fitted independently inside each fold during 5-fold Stratified Cross-Validation on the training subset ($N=100,778$).
3. **Equal Optimization Budget:** Exactly 25 Optuna TPE trials per model maximizing CV Macro-F1 (`random_state=42`).
4. **Strict Evaluation Hygiene:** The official test set (`KDDTest+.csv`, $N=22,543$) was evaluated exactly once at a fixed decision threshold of $0.50$.
5. **Statistical Rigor:** Paired McNemar tests on both overall accuracy and unseen attack detection, alongside 1,000-iteration bootstrap 95% confidence intervals.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Resolve project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
PRED_DIR = PROJECT_ROOT / 'results' / 'predictions'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'evaluation'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Define models and load artifacts
MODELS = {
    'majority_baseline_controlled': 'Majority Baseline',
    'logistic_regression_controlled': 'Logistic Regression',
    'decision_tree_controlled': 'Decision Tree',
    'random_forest_controlled': 'Random Forest'
}

PALETTE = {
    'Majority Baseline': '#8C8C8C',
    'Logistic Regression': '#1f77b4',
    'Decision Tree': '#2ca02c',
    'Random Forest': '#d62728'
}

metrics = {}
preds = {}
for key, label in MODELS.items():
    with open(METRICS_DIR / f'{key}.json', 'r') as f:
        metrics[label] = json.load(f)
    preds[label] = pd.read_csv(PRED_DIR / f'{key}_predictions.csv')

with open(METRICS_DIR / 'statistical_comparison.json', 'r') as f:
    stat_comp = json.load(f)

print(f'Successfully loaded artifacts for {len(MODELS)} models.')
print(f'Test set size: {len(preds["Random Forest"])} records.')


## 1. Overall Test Set Performance (Fixed Threshold = 0.50)

The table below compares overall classification metrics on the official `KDDTest+` benchmark ($N=22,543$).


In [ ]:
overall_rows = []
for label in MODELS.values():
    m = metrics[label]['overall_metrics']
    row = {
        'Model': label,
        'Accuracy (%)': m['accuracy'] * 100,
        'Macro-F1': m['macro_f1'],
        'Attack Recall (%)': m['attack_recall'] * 100,
        'Attack Precision (%)': m['attack_precision'] * 100,
        'Attack F1': m['attack_f1'],
        'FPR (False Alarm %)': m['false_positive_rate'] * 100,
        'FNR (Miss Rate %)': m['false_negative_rate'] * 100,
        'Specificity (%)': m['specificity'] * 100
    }
    overall_rows.append(row)

df_overall = pd.DataFrame(overall_rows).set_index('Model')
display(df_overall.round(2))


In [ ]:
# Grouped bar plot of primary performance indicators
plot_df = df_overall[['Macro-F1', 'Attack Recall (%)', 'FPR (False Alarm %)', 'Accuracy (%)']].reset_index()
plot_melted = plot_df.melt(id_vars='Model', var_name='Metric', value_name='Value')

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=plot_melted, x='Metric', y='Value', hue='Model', palette=PALETTE, ax=ax)
for p in ax.patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        ax.annotate(f'{h:.1f}', (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 3), textcoords='offset points')

ax.set_title('Overall Performance Metrics on NSL-KDD Test Set (Threshold = 0.50)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
ax.set_ylabel('Score / Percentage')
ax.legend(title='Model', loc='upper right')
plt.tight_layout()
fig.savefig(FIG_DIR / '01_overall_test_metrics.png', dpi=150)
plt.show()


## 2. Confusion Matrices

Side-by-side confusion matrices illustrating True Negatives (Normal detected), False Positives (False Alarms), False Negatives (Missed Attacks), and True Positives (Detected Attacks).


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
labels_order = list(MODELS.values())

for ax, label in zip(axes, labels_order):
    cm = np.array(metrics[label]['overall_metrics']['confusion_matrix'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Attack'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    rec = metrics[label]['overall_metrics']['attack_recall'] * 100
    fpr = metrics[label]['overall_metrics']['false_positive_rate'] * 100
    ax.set_title(f'{label}\nRecall: {rec:.1f}% | FPR: {fpr:.1f}%', fontsize=11, fontweight='bold')
    ax.grid(False)

plt.tight_layout()
fig.savefig(FIG_DIR / '02_confusion_matrices.png', dpi=150)
plt.show()


## 3. Attack Category Breakdown: DoS, Probe, R2L, and U2R

Intrusion attacks span four distinct technical families. We examine detection rates across each family to locate where models succeed and fail.


In [ ]:
cat_rows = []
for label in MODELS.values():
    cm_cats = metrics[label]['attack_category_recall']
    for cat_name, c_data in cm_cats.items():
        cat_rows.append({
            'Model': label,
            'Category': cat_name,
            'Total Samples': c_data['total_samples'],
            'Detected': c_data['detected'],
            'Recall (%)': c_data['recall'] * 100
        })

df_cats = pd.DataFrame(cat_rows)
pivot_cats = df_cats.pivot(index='Category', columns='Model', values='Recall (%)')
pivot_cats['Total Test Samples'] = [7458, 2421, 2887, 67]
display(pivot_cats.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))
sns.barplot(data=df_cats[df_cats['Model'] != 'Majority Baseline'], x='Category', y='Recall (%)', hue='Model', palette=PALETTE, ax=ax)
for p in ax.patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        ax.annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 3), textcoords='offset points')

ax.set_title('Detection Rate (Recall) by Attack Category', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_ylabel('Recall (%)')
ax.legend(title='Trained Model', loc='upper right')
plt.tight_layout()
fig.savefig(FIG_DIR / '03_category_recalls.png', dpi=150)
plt.show()


## 4. The Core Research Hypothesis: Seen vs. Unseen Attack Generalization

This section directly addresses our project question:  
**Can simpler models generalize better to unseen attack types than more complex models?**

- **Seen Attacks (9,083 test samples):** 21 attack types present in the training set (`KDDTrain+`).
- **Unseen Attacks (3,750 test samples):** 17 novel attack types absent from training but present in `KDDTest+`.
- **Generalization Gap:** $\text{Recall}_{\text{Seen}} - \text{Recall}_{\text{Unseen}}$ (percentage points drop).


In [ ]:
seen_unseen_rows = []
for label in MODELS.values():
    su = metrics[label]['seen_vs_unseen_recall']
    seen_unseen_rows.append({
        'Model': label,
        'Seen Recall (%)': su['seen_recall'] * 100,
        'Unseen Recall (%)': su['unseen_recall'] * 100,
        'Generalization Drop (pp)': su['generalization_drop_percentage_points']
    })

df_su = pd.DataFrame(seen_unseen_rows).set_index('Model')
display(df_su.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

# Panel 1: Seen vs Unseen Recall
df_plot_su = df_su.drop('Majority Baseline')[['Seen Recall (%)', 'Unseen Recall (%)']].reset_index()
df_melt_su = df_plot_su.melt(id_vars='Model', var_name='Attack Split', value_name='Recall (%)')
sns.barplot(data=df_melt_su, x='Model', y='Recall (%)', hue='Attack Split', palette=['#2b5c8f', '#e26d5c'], ax=axes[0])
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[0].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h),
                         ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
axes[0].set_title('Seen vs. Unseen Attack Recall', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].legend(loc='upper right')

# Panel 2: Generalization Drop
df_gap = df_su.drop('Majority Baseline')[['Generalization Drop (pp)']].reset_index()
sns.barplot(data=df_gap, x='Model', y='Generalization Drop (pp)', palette=PALETTE, ax=axes[1])
for p in axes[1].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[1].annotate(f'{h:.1f} pp', (p.get_x() + p.get_width() / 2., h),
                         ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
axes[1].set_title('Generalization Drop (Seen Recall − Unseen Recall)', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 65)
axes[1].set_ylabel('Drop (percentage points)')

plt.tight_layout()
fig.savefig(FIG_DIR / '04_seen_vs_unseen_generalization.png', dpi=150)
plt.show()


## 5. Granular Breakdown: Detection Rates on All 17 Unseen Attack Types

We inspect the exact sample counts, detected instances, and recall percentages across each of the 17 unseen attack signatures.
Notice that the bottom **TOTAL (SUM)** row exactly equals the aggregate unseen detected counts:  
- Logistic Regression: 1,578 / 3,750 ($42.08\%$)  
- Decision Tree: 2,001 / 3,750 ($53.36\%$)  
- Random Forest: 872 / 3,750 ($23.25\%$)


In [ ]:
unseen_attacks = list(metrics['Random Forest']['per_unseen_attack_recall'].keys())
u_rows = []
for u_att in unseen_attacks:
    cat = metrics['Random Forest']['per_unseen_attack_recall'][u_att]['category']
    tot = metrics['Random Forest']['per_unseen_attack_recall'][u_att]['total_samples']
    row = {'Attack Name': u_att, 'Category': cat, 'Samples': tot}
    for label in ['Logistic Regression', 'Decision Tree', 'Random Forest']:
        det = metrics[label]['per_unseen_attack_recall'][u_att]['detected']
        rec = metrics[label]['per_unseen_attack_recall'][u_att]['recall'] * 100
        row[f'{label} Det'] = det
        row[f'{label} Rec (%)'] = rec
    u_rows.append(row)

df_unseen_all = pd.DataFrame(u_rows).sort_values(by='Samples', ascending=False).set_index('Attack Name')

# Add verification sum row
totals = {'Category': 'ALL UNSEEN', 'Samples': int(df_unseen_all['Samples'].sum())}
for label in ['Logistic Regression', 'Decision Tree', 'Random Forest']:
    tot_det = int(df_unseen_all[f'{label} Det'].sum())
    totals[f'{label} Det'] = tot_det
    totals[f'{label} Rec (%)'] = round(tot_det / totals['Samples'] * 100.0, 2)

df_unseen_with_totals = pd.concat([df_unseen_all, pd.DataFrame([totals], index=['TOTAL (SUM)'])])
display(df_unseen_with_totals)


## 6. Statistical Significance & Bootstrap Uncertainty Analysis

We provide two distinct McNemar tests:
1. **Full Test Set ($N=22,543$):** Evaluates overall 0-1 classification accuracy / error rates.
2. **Unseen Attack Subset ($N=3,750$):** Evaluates detection rate concordance on novel attacks specifically.
Additionally, we compute paired 1,000-sample bootstrap differences with 95% confidence intervals.


In [ ]:
# 1. McNemar tests on Full Test Set (N=22,543)
mcn_full_rows = []
mcn_full_dict = stat_comp.get('mcnemar_tests_full_test_set_n22543', stat_comp.get('mcnemar_tests', {}))
for pair_key, res in mcn_full_dict.items():
    ct = res['contingency_table']
    pair_name = pair_key.replace('_', ' ').title()
    mcn_full_rows.append({
        'Comparison Pair': pair_name,
        'Both Correct': ct['both_correct'],
        'M1 Correct / M2 Wrong': ct['m1_correct_m2_wrong'],
        'M1 Wrong / M2 Correct': ct['m1_wrong_m2_correct'],
        'Both Wrong': ct['both_wrong'],
        'Disagreements': res['disagreements'],
        'Chi2 Stat': res['statistic'],
        'p-value': f"{res['p_value']:.4e}",
        'Significant (p < 0.01)': 'Yes (p < 0.001)' if res['significant_at_01'] else 'No'
    })
df_mcn_full = pd.DataFrame(mcn_full_rows).set_index('Comparison Pair')
print('--- Table 5A: McNemar Tests on Full Test Set (N=22,543: Overall Accuracy) ---')
display(df_mcn_full)


In [ ]:
# 2. McNemar tests strictly on Unseen Attacks (N=3,750: Novel Attack Detection)
mcn_uns_rows = []
mcn_uns_dict = stat_comp.get('mcnemar_tests_unseen_attacks_n3750', {})
for pair_key, res in mcn_uns_dict.items():
    ct = res['contingency_table']
    pair_name = pair_key.replace('_', ' ').title()
    mcn_uns_rows.append({
        'Comparison Pair': pair_name,
        'Both Detected': ct['both_correct'],
        'M1 Detected / M2 Missed': ct['m1_correct_m2_wrong'],
        'M1 Missed / M2 Detected': ct['m1_wrong_m2_correct'],
        'Both Missed': ct['both_wrong'],
        'Disagreements': res['disagreements'],
        'Chi2 Stat': res['statistic'],
        'p-value': f"{res['p_value']:.4e}",
        'Significant (p < 0.01)': 'Yes (p < 0.001)' if res['significant_at_01'] else 'No'
    })
df_mcn_uns = pd.DataFrame(mcn_uns_rows).set_index('Comparison Pair')
print('--- Table 5B: McNemar Tests on Unseen Attack Subset (N=3,750: Novel Attack Recall) ---')
display(df_mcn_uns)


In [ ]:
# 3. Paired Bootstrap Differences (1,000 resamples with paired test indices)
pair_boot_rows = []
paired_dict = stat_comp.get('paired_bootstrap_95_confidence_intervals_differences', {})
for comp_key, diffs in paired_dict.items():
    comp_label = comp_key.replace('_', ' ').upper()
    f1_diff = diffs['macro_f1']
    u_diff = diffs['unseen_recall_pp']
    pair_boot_rows.append({
        'Comparison': comp_label,
        'Δ Macro-F1 (95% CI)': f"{f1_diff['mean']:+.4f} [{f1_diff['ci_95_lower']:+.4f}, {f1_diff['ci_95_upper']:+.4f}]",
        'Δ Unseen Recall (pp, 95% CI)': f"{u_diff['mean']:+.2f} pp [{u_diff['ci_95_lower']:+.2f}, {u_diff['ci_95_upper']:+.2f}]",
        'P(Δ ≤ 0)': f"{u_diff.get('p_value_diff_le_0', 0.0):.4f}"
    })
df_paired_boot = pd.DataFrame(pair_boot_rows).set_index('Comparison')
print('--- Table 5C: Paired Bootstrap Differences (1,000 Resamples, 95% Confidence Intervals) ---')
display(df_paired_boot)


## 7. Computational Complexity & Deployment Latency


In [ ]:
timing_rows = []
for label in ['Logistic Regression', 'Decision Tree', 'Random Forest']:
    t = metrics[label]['timing']
    timing_rows.append({
        'Model': label,
        'Full Training Time (s)': t['training_time_seconds'],
        'Test Inference Time (s)': t['inference_time_seconds'],
        'Latency (μs / sample)': t['inference_us_per_sample']
    })
df_timing = pd.DataFrame(timing_rows).set_index('Model')
display(df_timing)


## 8. Bounded Academic Synthesis & Research Hypotheses

### 1. Does Random Forest Suffer from Signature Memorization?
**Evidence supports severe out-of-distribution degradation:** Random Forest achieves near-perfect validation saturation ($0.9987$ Macro-F1) and high seen-attack test recall ($75.91\%$), but suffers a massive **$52.66$ percentage point collapse** on novel attacks ($23.25\%$ unseen recall). Feature subspace projection (`max_features='sqrt'`) fragments decision paths into local combinations of training traffic artifacts, leaving the ensemble incapable of recognizing altered attack distributions.

### 2. Do Simpler Models Generalize Better?
**Empirically supported on this benchmark, with vital caveats:**
- **Decision Tree** detected **$53.36\%$** of unseen attacks ($2,001 / 3,750$) — more than $2.29\times$ higher than Random Forest ($23.25\%$), with a generalization drop of only $29.04$ pp.
- **Logistic Regression** detected **$42.08\%$** of unseen attacks ($1,578 / 3,750$) — nearly $1.81\times$ higher than Random Forest ($29.44$ pp drop).
- **Caveat 1 (Alert Fatigue):** Logistic Regression suffers a $7.55\%$ false positive rate ($733$ false alarms), nearly $3\times$ higher than Random Forest ($2.71\%$, $263$ false alarms).
- **Caveat 2 (Stealth Blindness):** On non-volumetric attacks (`mailbomb`, `processtable`, `worm`, `snmpgetattack`), both Logistic Regression and Random Forest collapsed to $< 1\%$ or $0\%$ recall.
- **Caveat 3 (No True Zero-Day Claim):** These 17 attacks are synthetic DARPA benchmark variations from 1999–2009; this does not constitute evidence of operational zero-day exploit defense.
